In [ ]:
"""
Download Analysis-Ready PlanetScope Sandbox data for Pará, Brazil
using Planet's Processing API.

Requirements:
    pip install requests

Set your API key before running:

Linux/macOS:
    export PLANET_API_KEY="your_api_key"

Windows PowerShell:
    $env:PLANET_API_KEY="your_api_key"
"""

import json
import os
from pathlib import Path

import requests


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

API_KEY = os.getenv("PLANET_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "PLANET_API_KEY is not set. Set it as an environment variable first."
    )

# Planet/Sentinel Hub Processing API endpoint.
# Use the endpoint assigned to your Planet account/region.
PROCESSING_URL = "https://services.sentinel-hub.com/api/v1/process"

# Sandbox BYOC collection supplied for the Pará ARPS region.
# Processing API collection identifiers generally use the byoc- prefix.
COLLECTION_ID = "byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e"

# Valid sandbox date range must be used.
TIME_FROM = "2022-01-01T00:00:00Z"
TIME_TO = "2022-01-31T23:59:59Z"

OUTPUT_DIR = Path("planet_output")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "para_arps_2022_01.tif"
REQUEST_FILE = OUTPUT_DIR / "processing_request.json"


# ---------------------------------------------------------------------
# Area of interest
# ---------------------------------------------------------------------

AOI = {
    "type": "Polygon",
    "coordinates": [
        [
            [-52.4825, -6.8735],
            [-52.4825, -6.65775],
            [-52.26725, -6.65775],
            [-52.26725, -6.8735],
            [-52.4825, -6.8735],
        ]
    ],
}


# ---------------------------------------------------------------------
# Processing API request
# ---------------------------------------------------------------------

request_body = {
    "input": {
        "bounds": {
            "properties": {
                "crs": "http://www.opengis.net/def/crs/EPSG/0/4326"
            },
            "geometry": AOI,
        },
        "data": [
            {
                "type": COLLECTION_ID,
                "dataFilter": {
                    "timeRange": {
                        "from": TIME_FROM,
                        "to": TIME_TO,
                    },
                    "mosaickingOrder": "mostRecent",
                },
                "processing": {
                    "upsampling": "BILINEAR",
                    "downsampling": "BILINEAR",
                },
            }
        ],
    },
    "output": {
        "width": 2048,
        "height": 2048,
        "responses": [
            {
                "identifier": "default",
                "format": {
                    "type": "image/tiff",
                },
            }
        ],
    },
    "evalscript": """
//VERSION=3

function setup() {
    return {
        input: [{
            bands: ["blue", "green", "red", "nir"],
            units: "REFLECTANCE"
        }],
        output: {
            bands: 4,
            sampleType: "FLOAT32"
        }
    };
}

function evaluatePixel(sample) {
    return [
        sample.blue,
        sample.green,
        sample.red,
        sample.nir
    ];
}
""",
}


# Save the request for inspection and debugging.
REQUEST_FILE.write_text(
    json.dumps(request_body, indent=2),
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Send request
# ---------------------------------------------------------------------

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "Accept": "image/tiff",
}

response = requests.post(
    PROCESSING_URL,
    headers=headers,
    json=request_body,
    timeout=300,
)


# ---------------------------------------------------------------------
# Handle response
# ---------------------------------------------------------------------

if response.ok:
    OUTPUT_FILE.write_bytes(response.content)

    print(f"Downloaded: {OUTPUT_FILE}")
    print(f"Size: {OUTPUT_FILE.stat().st_size:,} bytes")
    print(f"Request saved to: {REQUEST_FILE}")

else:
    print(f"Request failed with HTTP {response.status_code}")
    print(response.text)

    response.raise_for_status()
